In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import accuracy_score

In [2]:
train =pd.read_csv('./input/train.csv')
test = pd.read_csv('./input/test.csv')
sample_submission = pd.read_csv('./input/sample_submission.csv')

In [3]:
sample_submission.head()

,PassengerId,Transported
0,0013_01,False
1,0018_01,False
2,0019_01,False
3,0021_01,False
4,0023_01,False


In [4]:
train.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [5]:
test.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name
0,0013_01,Earth,True,G/3/S,TRAPPIST-1e,27.0,False,0.0,0.0,0.0,0.0,0.0,Nelly Carsoning
1,0018_01,Earth,False,F/4/S,TRAPPIST-1e,19.0,False,0.0,9.0,0.0,2823.0,0.0,Lerome Peckers
2,0019_01,Europa,True,C/0/S,55 Cancri e,31.0,False,0.0,0.0,0.0,0.0,0.0,Sabih Unhearfus
3,0021_01,Europa,False,C/1/S,TRAPPIST-1e,38.0,False,0.0,6652.0,0.0,181.0,585.0,Meratz Caltilter
4,0023_01,Earth,False,F/5/S,TRAPPIST-1e,20.0,False,10.0,0.0,635.0,0.0,0.0,Brence Harperez


In [6]:
data = pd.concat([train, test], sort=False).reset_index(drop=True)

In [7]:
data["HomePlanet"] = data["HomePlanet"].fillna("Earth").map({"Earth": 0, "Europa": 1, "Mars": 2})
data["CryoSleep"] = data["CryoSleep"].fillna(False).map({False: 0, True: 1})
data["Age"] = data["Age"].fillna(data["Age"].median())
data["VIP"] = data["VIP"].fillna(False).map({False: 0, True: 1})
data["Destination"]=data["Destination"].fillna("TRAPPIST-1e").map({"TRAPPIST-1e": 0, "55 Cancri e": 1, "PSO J318.5-22": 2}).astype(int)
data["Payment"] = data[["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]].fillna(0).sum(axis=1)
data[['Deck', 'Num', 'Side']] = data['Cabin'].str.split('/', expand=True)
data["Side"] = data["Side"].fillna("S").map({"S": 0, "P": 1}).astype(int)

/tmp/ipykernel_33401/1478014258.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data["CryoSleep"] = data["CryoSleep"].fillna(False).map({False: 0, True: 1})
/tmp/ipykernel_33401/1478014258.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data["VIP"] = data["VIP"].fillna(False).map({False: 0, True: 1})


In [8]:
print(data.columns.tolist())

['PassengerId', 'HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'Name', 'Transported', 'Payment', 'Deck', 'Num', 'Side']


In [9]:
delete_cols = ["Name", "PassengerId", "Cabin","Deck","Num","RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
data = data.drop(columns=delete_cols)

In [11]:
train = data[:len(train)]
test = data[len(train):]
y_train = train["Transported"].map({False: 0, True: 1})
X_train = train.drop(columns=["Transported"])
X_test = test.drop(columns=["Transported"])
categorical_features = ["HomePlanet", "CryoSleep", "VIP", "Destination", "Side"]


In [12]:

y_preds = []
models = []
oof_train = np.zeros((len(X_train),))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

params = {
    'objective': 'binary',
    'max_bin': 300,
    'learning_rate': 0.05,
    'num_leaves': 40
}

for fold_id, (train_index, valid_index) in enumerate(cv.split(X_train, y_train)):

    X_tr = X_train.loc[train_index, :]
    X_val = X_train.loc[valid_index, :]
    y_tr = y_train[train_index]
    y_val = y_train[valid_index]
    lgb_train = lgb.Dataset(X_tr, y_tr,
                                             categorical_feature=categorical_features)
    lgb_eval = lgb.Dataset(X_val, y_val, reference=lgb_train,
                                            categorical_feature=categorical_features)

    model = lgb.train(params, lgb_train,
                                   valid_sets=[lgb_train, lgb_eval],
                                   callbacks=[lgb.log_evaluation(10), lgb.early_stopping(10)],
                                   num_boost_round=1000)
    
    oof_train[valid_index] = model.predict(X_val, num_iteration=model.best_iteration)
    y_pred = model.predict(X_test, num_iteration=model.best_iteration)
    y_preds.append(y_pred)
    models.append(model)

[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003965 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 395
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
Training until validation scores don't improve for 10 rounds
[10]	training's binary_logloss: 0.587436	valid_1's binary_logloss: 0.59359
[20]	training's binary_logloss: 0.540119	valid_1's binary_logloss: 0.552137
[30]	training's binary_logloss: 0.515158	valid_1's binary_logloss: 0.53445
[40]	training's binary_logloss: 0.499665	valid_1's binary_logloss: 0.52622
[50]	training's binary_logloss: 0.489737	valid_1's binary_logloss: 0.52181
[60]	trainin

In [13]:
scores = [
    m.best_score['valid_1']['binary_logloss'] for m in models
]
score = sum(scores) / len(scores)
print('===CV scores===')
print(scores)
print(score)

===CV scores===
[np.float64(0.5197529883526537), np.float64(0.5304830585559389), np.float64(0.5153639112783782), np.float64(0.5261284860828042), np.float64(0.5231254677573136)]
0.5229707824054177


In [14]:
from sklearn.metrics import accuracy_score

y_pred_oof = (oof_train > 0.5).astype(int)
accuracy_score(y_train, y_pred_oof)

0.7391004256298171

In [15]:
y_pred = (y_pred > 0.5).astype(int)
y_pred[:10]


array([1, 0, 1, 0, 0, 0, 1, 1, 1, 0])

In [18]:
y_sub = sum(y_preds) / len(y_preds)
y_sub = (y_sub > 0.5).astype(int)
y_sub[:10]

array([1, 0, 1, 0, 0, 0, 1, 1, 1, 0])

In [ ]:
sub = pd.read_csv('./input/sample_submission.csv')
sub['Transported'] = y_sub.astype(bool)
sub.to_csv('s_t_submission_lightgbm_v3_Side.csv', index=False)

sub.head()
#public score:0.74000→0.74327→0.74304

,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,False
4,0023_01,False
